# Write a protein back out, and read it with openff-pablo unchanged

Hen lysozyme from Pablo's corpus, which has four disulfides. `save_pdb`
writes a `CONECT` for each disulfide and for nothing else, and Pablo reads
the file with no extra arguments.

In [ ]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [ ]:
import urllib.request
from pathlib import Path

cache = Path("../assets_cache")
cache.mkdir(exist_ok=True)
source = cache / "193l_prepared.pdb"
if not source.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/openforcefield/openff-pablo/main/"
        "openff/pablo/_tests/data/prepared_pdbs/193l_prepared.pdb",
        source,
    )

In [ ]:
from mbuild.biopolymers import Protein

mbuild_protein = Protein(source)
print(len(list(mbuild_protein.residues())), "residues,", mbuild_protein.n_particles, "atoms,", mbuild_protein.n_bonds, "bonds, net charge", mbuild_protein.net_formal_charge)
for record in mbuild_protein.bond_records():
    print(record["residue_names"], record["residue_numbers"], record["atom_names"], "leaving", record["leaving_atoms"])

In [ ]:
written = cache / "193l_mbuild.pdb"
mbuild_protein.save_pdb(written, overwrite=True)

lines = written.read_text().splitlines()
print({kind: sum(line.startswith(kind) for line in lines) for kind in ("ATOM", "HETATM", "TER", "CONECT")})
print(*[line for line in lines if line.startswith("CONECT")], sep="\n")

In [ ]:
from openff.pablo import topology_from_pdb

openff_topology = topology_from_pdb(written)
print(openff_topology.n_molecules, "molecule,", openff_topology.n_atoms, "atoms,", openff_topology.n_bonds, "bonds, net charge", openff_topology.molecule(0).total_charge)
assert openff_topology.n_bonds == mbuild_protein.n_bonds